# Experiment 7 — Feature Engineering

**Aim:** Engineer new features from the Titanic data (family size, alone flag, age groups), encode and scale the columns, select the best features, and measure how much model accuracy improves compared to the raw features.

**Description:** Feature engineering is the process of creating, transforming and selecting input features so that a model can learn better. It includes handling missing values, encoding categorical variables, scaling numerical values, and creating new features from existing ones. Good features often improve model performance more than choosing a more complicated algorithm.

**Dataset:** Titanic passenger data (local copy: `datasets/titanic_data.csv` — same file as Experiments 1 and 6).

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

for candidate in [Path("datasets"), Path("../datasets")]:
    if candidate.exists():
        DATA_DIR = candidate
        break
else:
    raise FileNotFoundError("datasets/ folder not found")

for candidate in [Path("outputs"), Path("../outputs")]:
    if candidate.exists():
        OUTPUT_DIR = candidate
        break

RES_DIR = OUTPUT_DIR / "results"
RES_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load the data and fill missing values

`Age` is filled with the **training median** and `Embarked` with the **training mode** — both statistics are computed on the training split only. The same train/test split (stratified, 80:20) is reused for the baseline and the engineered model, so the comparison is fair.

In [2]:
df = pd.read_csv(DATA_DIR / "titanic_data.csv")
print("Dataset shape:", df.shape)
print("\nMissing values in the columns we use:")
print(df[["Pclass", "Sex", "Age", "Fare", "SibSp", "Parch", "Embarked"]].isna().sum())

# One common split of the raw rows for both models
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=RANDOM_STATE, stratify=df["Survived"]
)

age_median = train_df["Age"].median()
embarked_mode = train_df["Embarked"].mode()[0]
print(f"\nTrain Age median = {age_median:.1f}, Train Embarked mode = '{embarked_mode}'")

for part in (train_df, test_df):
    part["Age"] = part["Age"].fillna(age_median)
    part["Embarked"] = part["Embarked"].fillna(embarked_mode)

y_train = train_df["Survived"]
y_test = test_df["Survived"]
print(f"Train size: {len(train_df)} | Test size: {len(test_df)}")

Dataset shape: (891, 12)

Missing values in the columns we use:
Pclass        0
Sex           0
Age         177
Fare          0
SibSp         0
Parch         0
Embarked      2
dtype: int64

Train Age median = 28.5, Train Embarked mode = 'S'
Train size: 712 | Test size: 179


## 2. Baseline model — raw columns only

`Pclass`, `Sex` (male = 0, female = 1), `Age` and `Fare`, with no new features.

In [3]:
X_train_base = train_df[["Pclass", "Sex", "Age", "Fare"]].copy()
X_test_base = test_df[["Pclass", "Sex", "Age", "Fare"]].copy()
X_train_base["Sex"] = X_train_base["Sex"].map({"male": 0, "female": 1})
X_test_base["Sex"] = X_test_base["Sex"].map({"male": 0, "female": 1})

base_clf = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
base_clf.fit(X_train_base, y_train)
base_acc = accuracy_score(y_test, base_clf.predict(X_test_base))
print(f"Baseline test accuracy (raw features): {base_acc:.4f}")

Baseline test accuracy (raw features): 0.7877


## 3. Create new features

- `family_size` = SibSp + Parch + 1 (the passenger counts as 1)
- `is_alone` = 1 when the passenger travels alone, else 0
- `age_group` = child (< 18), adult (18–60), senior (> 60)

In [4]:
def engineer_features(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    out["family_size"] = out["SibSp"] + out["Parch"] + 1
    out["is_alone"] = (out["family_size"] == 1).astype(int)
    out["age_group"] = pd.cut(
        out["Age"], bins=[0, 18, 60, 120], labels=["child", "adult", "senior"], include_lowest=True
    )
    return out


train_eng = engineer_features(train_df)
test_eng = engineer_features(test_df)

print("New features (first 5 training rows):")
print(train_eng[["family_size", "is_alone", "age_group"]].head())
print("\nAge group counts (train):")
print(train_eng["age_group"].value_counts())

New features (first 5 training rows):
     family_size  is_alone age_group
692            1         1     adult
481            1         1     adult
527            1         1     adult
855            2         0     child
801            3         0     adult

Age group counts (train):
age_group
adult     588
child     108
senior     16
Name: count, dtype: int64


## 4. One-hot encoding, scaling and feature selection

Categorical columns (`Sex`, `Embarked`, `age_group`) are one-hot encoded. `Age` and `Fare` are standardised with a `StandardScaler` **fit on the training data only**. `SelectKBest` then ranks the features by their ANOVA F-score against the target.

In [5]:
cat_cols = ["Sex", "Embarked", "age_group"]
num_cols = ["Age", "Fare"]
keep_cols = ["Pclass", "SibSp", "Parch", "family_size", "is_alone"]

def build_matrix(frame: pd.DataFrame, scaler: StandardScaler | None, fit: bool):
    cats = pd.get_dummies(frame[cat_cols], columns=cat_cols, drop_first=True).reset_index(drop=True)
    if fit:
        nums = scaler.fit_transform(frame[num_cols])
    else:
        nums = scaler.transform(frame[num_cols])
    nums_df = pd.DataFrame(nums, columns=num_cols)
    rest = frame[keep_cols].reset_index(drop=True)
    return pd.concat([nums_df, rest, cats], axis=1), scaler

scaler = StandardScaler()
X_train_eng, scaler = build_matrix(train_eng, scaler, fit=True)
X_test_eng, _ = build_matrix(test_eng, scaler, fit=False)

selector = SelectKBest(score_func=f_classif, k=8)
selector.fit(X_train_eng, y_train)

scores = pd.DataFrame({
    "feature": X_train_eng.columns,
    "F-score": selector.scores_,
}).sort_values("F-score", ascending=False)
print("Feature ranking (ANOVA F-score):")
print(scores.to_string(index=False))

selected = X_train_eng.columns[selector.get_support()].tolist()
print("\nSelected features (k=8):", selected)

Feature ranking (ANOVA F-score):


         feature    F-score
        Sex_male 306.593249
          Pclass  97.835853
            Fare  58.314907
        is_alone  34.601859
      Embarked_S  23.546389
           Parch   5.066904
 age_group_adult   4.531142
             Age   3.779379
age_group_senior   2.660201
      Embarked_Q   0.705233
     family_size   0.516703
           SibSp   0.484537

Selected features (k=8): ['Age', 'Fare', 'Pclass', 'Parch', 'is_alone', 'Sex_male', 'Embarked_S', 'age_group_adult']


## 5. Train the model on the engineered features and compare

In [6]:
eng_clf = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
eng_clf.fit(X_train_eng[selected], y_train)
eng_acc = accuracy_score(y_test, eng_clf.predict(X_test_eng[selected]))

results = pd.DataFrame([
    {"Model": "Baseline (Pclass, Sex, Age, Fare)", "Test accuracy": base_acc},
    {"Model": "Engineered features (SelectKBest k=8)", "Test accuracy": eng_acc},
])
results["Improvement"] = results["Test accuracy"] - base_acc
print(results.to_string(index=False))

results.to_csv(RES_DIR / "exp07_accuracy_comparison.csv", index=False)
print(f"\nSaved to {RES_DIR / 'exp07_accuracy_comparison.csv'}")

                                Model  Test accuracy  Improvement
    Baseline (Pclass, Sex, Age, Fare)       0.787709     0.000000
Engineered features (SelectKBest k=8)       0.782123    -0.005587

Saved to outputs\results\exp07_accuracy_comparison.csv


## Observations / Conclusion

Based on the run above:

1. `Sex_male` dominates the ANOVA ranking (F = 306.6), followed by Pclass (97.8) and Fare (58.3). Among the engineered features, **`is_alone` ranked 4th (F = 34.6)** and was selected, while `family_size` itself ranked near the bottom (F = 0.52) — the alone/not-alone flag already carries most of the family information, and raw sibling/spouse counts add little on their own.
2. SelectKBest kept 8 of 12 features: Age, Fare, Pclass, Parch, is_alone, Sex_male, Embarked_S and age_group_adult.
3. The engineered model scored **78.2% vs 78.8% for the baseline** — a change of −0.6%, which is about **one passenger** on the 179-person test set, i.e. within noise. The exercise shows the honest lesson of feature engineering: new features are not automatically better; every change must be validated against a proper train/test comparison.
4. The small drop likely comes from the linear model: one-hot age groups and scaled fares add dimensions without adding strong signal beyond what Sex/Pclass/Fare already provide. Feature *interactions* (e.g. sex × class) would be the natural next attempt.